#Corporate Market Research & Strategy without api

Scenario: Corporate Market Research & Strategy

A company wants to explore launching a new product in a competitive market. The Manager Agent oversees the process and delegates tasks to specialized workers.

👩‍💼 Manager Agent
- Receives the overall goal: “Evaluate feasibility of launching Product Y in Asia.”
- Dynamically assigns tasks to worker agents depending on what’s needed.
- Example: If budget is unclear → send to Finance Worker. If regulations are complex → send to Legal Worker.

📊 Worker Agents

- Market Research Worker
- Collects competitor data, customer preferences, and demand forecasts.
- Reports: “High demand in Tier‑1 cities, moderate competition.”
- Finance Worker
- Analyzes budget, ROI, and pricing strategy.
- Reports: “Estimated $10M investment, ROI in 2 years.”
- Operations Worker
- Evaluates supply chain, production capacity, and logistics.
- Reports: “Factories can scale up, but shipping costs are high.”
- Legal Worker
- Reviews compliance, intellectual property, and regional regulations.
- Reports: “Trademark available, but import laws require certification.”
- HR Worker
- Assesses staffing needs and training requirements.
- Reports: “Need 50 new hires for customer support and sales.”

In [ ]:
# WORKER AGENTS
def market_research_worker():
    print("\n[Market Research Worker] Analyzing market...")
    return {
        "demand": "high",
        "competition": "moderate",
        "insight": "High demand in Tier-1 cities"
    }


def finance_worker():
    print("\n[Finance Worker] Evaluating financials...")
    return {
        "investment": 10_000_000,
        "roi_years": 2,
        "feasible": True
    }


def operations_worker():
    print("\n[Operations Worker] Checking operations...")
    return {
        "production": "scalable",
        "logistics": "expensive",
        "risk": "medium"
    }


def legal_worker():
    print("\n[Legal Worker] Checking regulations...")
    return {
        "trademark": "available",
        "compliance": "certification required",
        "risk": "medium"
    }


def hr_worker():
    print("\n[HR Worker] Evaluating staffing...")
    return {
        "new_hires": 50,
        "training_required": True,
        "cost_level": "medium"
    }


# MANAGER AGENT
def manager_agent(goal):
    print("\n[Manager Agent] Received goal:", goal)

    results = {}

    # Dynamic decision-making (rule-based)
    if "market" in goal.lower() or "launch" in goal.lower():
        results["market"] = market_research_worker()

    # Always check finance for feasibility
    results["finance"] = finance_worker()

    # If expansion → need operations
    if "asia" in goal.lower():
        results["operations"] = operations_worker()

    # If regulations matter → legal
    if "asia" in goal.lower():
        results["legal"] = legal_worker()

    # If scaling → HR needed
    results["hr"] = hr_worker()

    return results


# FINAL DECISION AGENT
def strategy_decision_agent(all_reports):
    print("\n[Strategy Agent] Making final business decision...")

    market = all_reports.get("market", {})
    finance = all_reports.get("finance", {})
    operations = all_reports.get("operations", {})
    legal = all_reports.get("legal", {})
    hr = all_reports.get("hr", {})

    # Simple decision logic
    if market.get("demand") == "high" and finance.get("feasible"):
        if operations.get("logistics") == "expensive" or legal.get("risk") == "medium":
            decision = "Proceed with caution"
        else:
            decision = "Launch immediately"
    else:
        decision = "Not recommended"

    return {
        "decision": decision,
        "summary": {
            "market": market,
            "finance": finance,
            "operations": operations,
            "legal": legal,
            "hr": hr
        }
    }


# MAIN SYSTEM
def corporate_multi_agent(goal):
    print("Goal:", goal)

    # Step 1: Manager assigns tasks
    reports = manager_agent(goal)

    # Step 2: Strategy decision
    final_decision = strategy_decision_agent(reports)

    return final_decision


# RUN
response = corporate_multi_agent(
    "Evaluate feasibility of launching Product Y in Asia"
)

print("\n===== FINAL OUTPUT =====")
print(response)

Goal: Evaluate feasibility of launching Product Y in Asia

[Manager Agent] Received goal: Evaluate feasibility of launching Product Y in Asia

[Market Research Worker] Analyzing market...

[Finance Worker] Evaluating financials...

[Operations Worker] Checking operations...

[Legal Worker] Checking regulations...

[HR Worker] Evaluating staffing...

[Strategy Agent] Making final business decision...

===== FINAL OUTPUT =====
{'decision': 'Proceed with caution', 'summary': {'market': {'demand': 'high', 'competition': 'moderate', 'insight': 'High demand in Tier-1 cities'}, 'finance': {'investment': 10000000, 'roi_years': 2, 'feasible': True}, 'operations': {'production': 'scalable', 'logistics': 'expensive', 'risk': 'medium'}, 'legal': {'trademark': 'available', 'compliance': 'certification required', 'risk': 'medium'}, 'hr': {'new_hires': 50, 'training_required': True, 'cost_level': 'medium'}}}


#Using API

In [ ]:
# SETUP
from groq import Groq
from google.colab import userdata
import json
api_key = userdata.get("newapi")

if not api_key:
    raise ValueError("API key not found in Colab Secrets")

client = Groq(api_key=api_key)

MODEL_NAME = "llama-3.3-70b-versatile"
def clean_response(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.replace("```json", "").replace("```", "").strip()

    return text


def safe_parse_json(text):
    try:
        return json.loads(text)
    except:
        return {
            "error": "Invalid JSON",
            "raw_output": text
        }


def call_llm(prompt):
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Return ONLY valid JSON. No explanation, no markdown."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        content = response.choices[0].message.content

        if not content or content.strip() == "":
            return {"error": "Empty response from model"}
        content = clean_response(content)
        return safe_parse_json(content)

    except Exception as e:
        return {"error": str(e)}


# WORKER AGENTS
def market_research_worker(goal):
    print("\n[Market Research Worker]")

    prompt = f"""
    Analyze market for: {goal}

    Output strictly in JSON:
    {{
      "demand": "high/medium/low",
      "competition": "high/medium/low",
      "insight": "short summary"
    }}
    """
    return call_llm(prompt)


def finance_worker(goal):
    print("\n[Finance Worker]")

    prompt = f"""
    Analyze financial feasibility for: {goal}

    Output strictly in JSON:
    {{
      "investment": number,
      "roi_years": number,
      "feasible": true/false
    }}
    """
    return call_llm(prompt)


def operations_worker(goal):
    print("\n[Operations Worker]")

    prompt = f"""
    Evaluate operations for: {goal}

    Output strictly in JSON:
    {{
      "production": "scalable/not scalable",
      "logistics": "cheap/moderate/expensive",
      "risk": "low/medium/high"
    }}
    """
    return call_llm(prompt)


def legal_worker(goal):
    print("\n[Legal Worker]")

    prompt = f"""
    Evaluate legal and compliance for: {goal}

    Output strictly in JSON:
    {{
      "trademark": "available/not available",
      "compliance": "short description",
      "risk": "low/medium/high"
    }}
    """
    return call_llm(prompt)


def hr_worker(goal):
    print("\n[HR Worker]")

    prompt = f"""
    Evaluate HR requirements for: {goal}

    Output strictly in JSON:
    {{
      "new_hires": number,
      "training_required": true/false,
      "cost_level": "low/medium/high"
    }}
    """
    return call_llm(prompt)


# MANAGER AGENT
def manager_agent(goal):
    print("\n[Manager Agent] Deciding which workers to call...")

    results = {}

    # Always required
    results["market"] = market_research_worker(goal)
    results["finance"] = finance_worker(goal)

    # Conditional delegation
    if "asia" in goal.lower():
        results["operations"] = operations_worker(goal)
        results["legal"] = legal_worker(goal)

    # Always check HR
    results["hr"] = hr_worker(goal)

    return results


# STRATEGY DECISION AGENT
def strategy_decision_agent(goal, reports):
    print("\n[Strategy Agent] Making final decision...")

    prompt = f"""
    You are a senior business strategist.

    Goal:
    {goal}

    Reports:
    {json.dumps(reports, indent=2)}

    Based on all reports, provide final decision.

    Output strictly in JSON:
    {{
      "decision": "Launch / Proceed with caution / Not recommended",
      "reason": "clear explanation"
    }}
    """

    return call_llm(prompt)


# MAIN SYSTEM
def corporate_multi_agent(goal):
    print("Goal:", goal)

    # Step 1: Manager assigns tasks
    reports = manager_agent(goal)

    # Step 2: Final strategy decision
    decision = strategy_decision_agent(goal, reports)

    return {
        "reports": reports,
        "final_decision": decision
    }


# RUN
response = corporate_multi_agent(
    "Evaluate feasibility of launching Product Y in Asia"
)

print("\n===== FINAL OUTPUT =====")
print(json.dumps(response, indent=2))

Goal: Evaluate feasibility of launching Product Y in Asia

[Manager Agent] Deciding which workers to call...

[Market Research Worker]

[Finance Worker]

[Operations Worker]

[Legal Worker]

[HR Worker]

[Strategy Agent] Making final decision...

===== FINAL OUTPUT =====
{
  "reports": {
    "market": {
      "demand": "high",
      "competition": "medium",
      "insight": "Growing middle class and increasing consumer spending in Asia make it an attractive market for Product Y"
    },
    "finance": {
      "investment": 1000000,
      "roi_years": 5,
      "feasible": true
    },
    "operations": {
      "production": "scalable",
      "logistics": "moderate",
      "risk": "medium"
    },
    "legal": {
      "trademark": "available",
      "compliance": "Product Y meets local regulations and standards in Asia, including those related to product safety, labeling, and packaging",
      "risk": "medium"
    },
    "hr": {
      "new_hires": 10,
      "training_required": true,
      